# 01 Download and preprocess (CPU, Internet on)
Run once per collection. **This notebook has not been run against the real dataset.** Check the folder
layout and the table's column names in the first cells and adjust `mmonc.data.COLUMN_MAP` if needed.
Add a Kaggle secret `SYNAPSE_AUTH_TOKEN`.

In [ ]:
# Kaggle: turn Internet ON. Add secrets GITHUB_REPO (owner/name) and, if private, GITHUB_TOKEN.
import os, subprocess, sys
try:
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    repo, tok = sec.get_secret("GITHUB_REPO"), None
    try: tok = sec.get_secret("GITHUB_TOKEN")
    except Exception: pass
    url = f"https://{tok + '@' if tok else ''}github.com/{repo}.git"
    if not os.path.exists("/kaggle/working/repo"):
        subprocess.run(["git", "clone", "--depth", "1", url, "/kaggle/working/repo"], check=True)
    os.chdir("/kaggle/working/repo")
except ImportError:
    pass  # running locally from the repository root
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
sys.path.insert(0, "src")

In [ ]:
!pip install -q synapseclient SimpleITK openpyxl
COLLECTION = "ISPY2"          # ISPY2, DUKE, ISPY1 or NACT
RAW, OUT, SIZE = "/kaggle/tmp/mamamia", "/kaggle/working/prepared", 64
from kaggle_secrets import UserSecretsClient
os.environ["SYNAPSE_AUTH_TOKEN"] = UserSecretsClient().get_secret("SYNAPSE_AUTH_TOKEN")

In [ ]:
# Step 1: list the Synapse project first; download only the table and this collection's folders.
import synapseclient, synapseutils
syn = synapseclient.login(authToken=os.environ["SYNAPSE_AUTH_TOKEN"])
for dirpath, dirnames, filenames in synapseutils.walk(syn, "syn60868042"):
    print(dirpath, len(dirnames), len(filenames)); break
# TODO after inspecting the listing: download the clinical table and the images/segmentations
# for COLLECTION into RAW (synapseutils.syncFromSynapse on the relevant sub-folders).

In [ ]:
import glob, numpy as np, pandas as pd
from mmonc.data import harmonise_table, preprocess_case
from mmonc.radiomics import simple_features, SIMPLE_NAMES
raw = pd.read_excel(glob.glob(f"{RAW}/**/*clinical*imaging*.xlsx", recursive=True)[0])
print(list(raw.columns))                       # check against COLUMN_MAP
tb = harmonise_table(raw)
tb = tb[tb.collection == COLLECTION].reset_index(drop=True)
os.makedirs(OUT, exist_ok=True); rad, ok = {}, []
for cid in tb.case_id:
    if os.path.exists(f"{OUT}/{cid}.npz"): ok.append(cid); continue      # resume
    phases = sorted(glob.glob(f"{RAW}/**/images/{cid}/{cid}_000*.nii.gz", recursive=True))
    mask = glob.glob(f"{RAW}/**/segmentations/expert/{cid}.nii.gz", recursive=True)
    try:
        img, m = preprocess_case(phases, mask[0], size=SIZE)
        np.savez_compressed(f"{OUT}/{cid}.npz", image=img, mask=m); ok.append(cid)
    except Exception as e:
        print("skipped", cid, e)
for cid in ok:
    z = np.load(f"{OUT}/{cid}.npz"); rad[cid] = simple_features(z["image"].astype("float32"), z["mask"])
pd.DataFrame.from_dict(rad, orient="index", columns=SIMPLE_NAMES).to_csv(f"{OUT}/radiomics_{COLLECTION}.csv")
tb[tb.case_id.isin(ok)].to_csv(f"{OUT}/table_{COLLECTION}.csv", index=False)
print(len(ok), "of", len(tb), "cases prepared")

After all four collections: concatenate the `table_*.csv` and `radiomics_*.csv` files into `table.csv` and
`radiomics.csv`, parse acquisition times into `dt0, dt1, dt2` (seconds from contrast injection), and publish the
folder as a private Kaggle dataset named `mamamia-crops`. Replace `simple_features` with
`mmonc.radiomics.pyradiomics_features` for the paper.